In [1]:
import os
import json
import io
import zipfile
import xml.etree.ElementTree as ET

import pandas as pd
import requests
from dotenv import load_dotenv

# 프로젝트 루트의 환경변수 파일 불러오기
load_dotenv("/Users/anna/bulsae/.env")

# API 키는 출력하지 않고 변수에만 저장
dart_api_key = os.getenv("DART_API_KEY")

if not dart_api_key:
    raise ValueError(".env 파일에서 DART_API_KEY를 찾을 수 없습니다.")

print("DART API 키를 확인했습니다.")

DART API 키를 확인했습니다.


In [2]:
# 공통 기업 목록 파일
file_path = "/Users/anna/bulsae/data/shared/companies.jsonl"

with open(file_path, encoding="utf-8") as file:
    companies = [
        json.loads(line)
        for line in file
        if line.strip()
    ]

companies_df = pd.DataFrame(companies)

print(f"불러온 기업 수: {len(companies_df)}개")
display(companies_df)

불러온 기업 수: 7개


,company_id,input_name,legal_name,dart_corp_code
0,COMPANY-INNISFREE,이니스프리,주식회사 이니스프리,00812353
1,COMPANY-ESPOIR,에스쁘아,주식회사 에스쁘아,01061150
2,COMPANY-AMOREPACIFIC,아모레퍼시픽,주식회사 아모레퍼시픽,00583424
3,COMPANY-ETUDE,에뛰드,주식회사 에뛰드,00183808
4,COMPANY-PACIFICTECH,퍼시픽테크,주식회사 퍼시픽테크,01784251
5,COMPANY-AMOSPROFESSIONAL,아모스프로페셔널,주식회사 아모스프로페셔널,00144456
6,COMPANY-COSRX,코스알엑스,주식회사 코스알엑스,01364491


In [3]:
required_columns = ["company_id", "dart_corp_code"]

missing_columns = [
    column
    for column in required_columns
    if column not in companies_df.columns
]

if missing_columns:
    print("필요한 컬럼이 없습니다:", missing_columns)
else:
    print("재무 조회에 필요한 컬럼을 확인했습니다.")
    display(companies_df[required_columns])

재무 조회에 필요한 컬럼을 확인했습니다.


,company_id,dart_corp_code
0,COMPANY-INNISFREE,00812353
1,COMPANY-ESPOIR,01061150
2,COMPANY-AMOREPACIFIC,00583424
3,COMPANY-ETUDE,00183808
4,COMPANY-PACIFICTECH,01784251
5,COMPANY-AMOSPROFESSIONAL,00144456
6,COMPANY-COSRX,01364491


In [4]:
# financials.jsonl에 사용할 컬럼
financial_columns = [
    "financial_id",
    "company_id",
    "fiscal_year",
    "period",
    "statement_scope",
    "amount_basis",
    "revenue_krw",
    "operating_profit_krw",
    "source",
    "verification_status",
]

# 아직 데이터가 없는 빈 재무 데이터 표
financials_df = pd.DataFrame(columns=financial_columns)

print(f"현재 재무 데이터: {len(financials_df)}건")
display(financials_df)

현재 재무 데이터: 0건


,financial_id,company_id,fiscal_year,period,statement_scope,amount_basis,revenue_krw,operating_profit_krw,source,verification_status


In [5]:
# 아모레퍼시픽의 2025년 연간 개별 재무제표 조회
url = "https://opendart.fss.or.kr/api/fnlttSinglAcntAll.json"

params = {
    "crtfc_key": dart_api_key,
    "corp_code": "00583424",  # 아모레퍼시픽 DART 고유번호
    "bsns_year": "2025",
    "reprt_code": "11011",    # 사업보고서
    "fs_div": "OFS",          # 개별 재무제표
}

response = requests.get(url, params=params, timeout=30)
response.raise_for_status()

data = response.json()

if data.get("status") != "000":
    raise ValueError(
        f'DART 조회 실패: {data.get("status")} - {data.get("message")}'
    )

# 응답 전체를 표로 변환
dart_financial_df = pd.DataFrame(data.get("list", []))

# 매출액과 영업이익 행 확인
selected_financial_df = dart_financial_df[
    dart_financial_df["account_nm"].isin(["매출액", "영업이익"])
].copy()

display(
    selected_financial_df[
        [
            "sj_nm",
            "account_nm",
            "thstrm_amount",
            "currency",
            "rcept_no",
        ]
    ]
)

,sj_nm,account_nm,thstrm_amount,currency,rcept_no
46,포괄손익계산서,영업이익,212514854678,KRW,20260318000785
62,포괄손익계산서,매출액,2653725793231,KRW,20260318000785


In [6]:
# 필요한 두 항목을 각각 하나씩 찾기
revenue_rows = selected_financial_df[
    selected_financial_df["account_nm"] == "매출액"
]

profit_rows = selected_financial_df[
    selected_financial_df["account_nm"] == "영업이익"
]

if len(revenue_rows) != 1 or len(profit_rows) != 1:
    raise ValueError(
        "매출액 또는 영업이익이 하나로 특정되지 않습니다. "
        "위 조회 결과를 확인해 주세요."
    )

revenue_row = revenue_rows.iloc[0]
profit_row = profit_rows.iloc[0]

if revenue_row["currency"] != "KRW" or profit_row["currency"] != "KRW":
    raise ValueError("통화가 KRW인지 확인해 주세요.")

if revenue_row["rcept_no"] != profit_row["rcept_no"]:
    raise ValueError("두 항목의 공시 접수번호가 다릅니다.")

# 쉼표가 있는 금액을 정수로 변환
def parse_dart_amount(value):
    return int(str(value).replace(",", "").strip())


receipt_no = revenue_row["rcept_no"]

amore_financial = {
    "financial_id": "FINANCIAL-AMOREPACIFIC-2025-ANNUAL-OFS",
    "company_id": "COMPANY-AMOREPACIFIC",
    "fiscal_year": 2025,
    "period": "연간",
    "statement_scope": "개별",
    "amount_basis": None,  # 기준 확인 후 입력
    "revenue_krw": parse_dart_amount(revenue_row["thstrm_amount"]),
    "operating_profit_krw": parse_dart_amount(profit_row["thstrm_amount"]),
    "source": [f"https://dart.fss.or.kr/dsaf001/main.do?rcpNo={receipt_no}"],
    "verification_status": "원문 확인 필요",
}

# 같은 셀을 다시 실행해도 동일한 기록이 중복되지 않도록 처리
financials_df = financials_df[
    financials_df["financial_id"] != amore_financial["financial_id"]
]

financials_df = pd.concat(
    [
        financials_df,
        pd.DataFrame([amore_financial], columns=financial_columns),
    ],
    ignore_index=True,
)

display(financials_df)

,financial_id,company_id,fiscal_year,period,statement_scope,amount_basis,revenue_krw,operating_profit_krw,source,verification_status
0,FINANCIAL-AMOREPACIFIC-2025-ANNUAL-OFS,COMPANY-AMOREPACIFIC,2025,연간,개별,None,2653725793231,212514854678,[https://dart.fss.or.kr/dsaf001/main.do?rcpNo=...,원문 확인 필요


In [7]:
# 사용하지 않을 컬럼 제거
financials_df = financials_df.drop(
    columns=["amount_basis", "verification_status"],
    errors="ignore",
)

# 이후 데이터를 추가할 때 사용할 컬럼 목록도 갱신
financial_columns = financials_df.columns.tolist()

display(financials_df)

,financial_id,company_id,fiscal_year,period,statement_scope,revenue_krw,operating_profit_krw,source
0,FINANCIAL-AMOREPACIFIC-2025-ANNUAL-OFS,COMPANY-AMOREPACIFIC,2025,연간,개별,2653725793231,212514854678,[https://dart.fss.or.kr/dsaf001/main.do?rcpNo=...


In [9]:
financials_df.head(10)

,financial_id,company_id,fiscal_year,period,statement_scope,revenue_krw,operating_profit_krw,source
0,FINANCIAL-AMOREPACIFIC-2025-ANNUAL-OFS,COMPANY-AMOREPACIFIC,2025,연간,개별,2653725793231,212514854678,[https://dart.fss.or.kr/dsaf001/main.do?rcpNo=...


In [10]:
# 기존 company_info.ipynb에 저장된 2025년 재무 조회 결과
# 금액 단위: 원

financial_candidates = [
    {
        "company_id": "COMPANY-INNISFREE",
        "company_name": "이니스프리",
        "revenue_krw": 209_848_553_422,
        "operating_profit_krw": 13_348_911_731,
        "statement_scope": None,
        "receipt_no": "20260401003769",
        "review_note": "금액 확인됨 · 재무제표 범위 확인 필요",
    },
    {
        "company_id": "COMPANY-ESPOIR",
        "company_name": "에스쁘아",
        "revenue_krw": 77_210_868_308,
        "operating_profit_krw": -452_624_866,
        "statement_scope": None,
        "receipt_no": "20260326001214",
        "review_note": "금액 확인됨 · 재무제표 범위 확인 필요",
    },
    {
        "company_id": "COMPANY-ETUDE",
        "company_name": "에뛰드",
        "revenue_krw": 115_092_257_274,
        "operating_profit_krw": 9_971_025_760,
        "statement_scope": None,
        "receipt_no": "20260331004546",
        "review_note": "감사보고서의 금액과 단위 확인 필요",
    },
    {
        "company_id": "COMPANY-AMOSPROFESSIONAL",
        "company_name": "아모스프로페셔널",
        "revenue_krw": 79_136_115_848,
        "operating_profit_krw": 6_982_117_086,
        "statement_scope": None,
        "receipt_no": "20260326000513",
        "review_note": "감사보고서의 금액과 단위 확인 필요",
    },
    {
        "company_id": "COMPANY-COSRX",
        "company_name": "코스알엑스",
        "revenue_krw": 456_246_005_204,
        "operating_profit_krw": 114_015_269_459,
        "statement_scope": None,
        "receipt_no": "20260402000034",
        "review_note": "감사보고서의 금액과 단위 확인 필요",
    },
    {
        "company_id": "COMPANY-PACIFICTECH",
        "company_name": "퍼시픽테크",
        "revenue_krw": 1_854_000_000,
        "operating_profit_krw": -922_000_000,
        "statement_scope": "개별",
        "receipt_no": "20260601000936",
        "review_note": "공시의 백만원 단위를 원으로 변환한 값",
    },
]

financial_candidates_df = pd.DataFrame(financial_candidates)

# 공시 원문 링크 추가
financial_candidates_df["report_url"] = (
    "https://dart.fss.or.kr/dsaf001/main.do?rcpNo="
    + financial_candidates_df["receipt_no"]
)

print(f"확인할 기업 수: {len(financial_candidates_df)}개")
display(financial_candidates_df)

확인할 기업 수: 6개


,company_id,company_name,revenue_krw,operating_profit_krw,statement_scope,receipt_no,review_note,report_url
0,COMPANY-INNISFREE,이니스프리,209848553422,13348911731,NaN,20260401003769,금액 확인됨 · 재무제표 범위 확인 필요,https://dart.fss.or.kr/dsaf001/main.do?rcpNo=2...
1,COMPANY-ESPOIR,에스쁘아,77210868308,-452624866,NaN,20260326001214,금액 확인됨 · 재무제표 범위 확인 필요,https://dart.fss.or.kr/dsaf001/main.do?rcpNo=2...
2,COMPANY-ETUDE,에뛰드,115092257274,9971025760,NaN,20260331004546,감사보고서의 금액과 단위 확인 필요,https://dart.fss.or.kr/dsaf001/main.do?rcpNo=2...
3,COMPANY-AMOSPROFESSIONAL,아모스프로페셔널,79136115848,6982117086,NaN,20260326000513,감사보고서의 금액과 단위 확인 필요,https://dart.fss.or.kr/dsaf001/main.do?rcpNo=2...
4,COMPANY-COSRX,코스알엑스,456246005204,114015269459,NaN,20260402000034,감사보고서의 금액과 단위 확인 필요,https://dart.fss.or.kr/dsaf001/main.do?rcpNo=2...
5,COMPANY-PACIFICTECH,퍼시픽테크,1854000000,-922000000,개별,20260601000936,공시의 백만원 단위를 원으로 변환한 값,https://dart.fss.or.kr/dsaf001/main.do?rcpNo=2...


In [11]:
# 검토한 6개 기업의 재무 데이터를 스키마에 맞게 변환
new_financials = []

for _, row in financial_candidates_df.iterrows():
    company_id = row["company_id"]

    # COMPANY-INNISFREE → INNISFREE
    company_key = company_id.removeprefix("COMPANY-")

    # 재무제표 범위를 아직 확인하지 못한 경우 UNKNOWN으로 구분
    scope_key = "OFS" if row["statement_scope"] == "개별" else "UNKNOWN"

    financial_record = {
        "financial_id": f"FINANCIAL-{company_key}-2025-ANNUAL-{scope_key}",
        "company_id": company_id,
        "fiscal_year": 2025,
        "period": "연간",
        "statement_scope": row["statement_scope"],
        "revenue_krw": int(row["revenue_krw"]),
        "operating_profit_krw": int(row["operating_profit_krw"]),

        # 출처는 객체를 담은 배열로 저장
        "source": [
            {
                "source_type": "disclosure",
                "source_name": (
                    "대규모기업집단현황공시"
                    if company_id == "COMPANY-PACIFICTECH"
                    else "감사보고서"
                ),
                "original_url": row["report_url"],
                "collected_at": None,
                "as_of_date": "2025-12-31",
                "evidence_location": (
                    "회사 손익현황"
                    if company_id == "COMPANY-PACIFICTECH"
                    else "손익계산서"
                ),
            }
        ],
    }

    new_financials.append(financial_record)

# 리스트를 데이터프레임으로 변환
new_financials_df = pd.DataFrame(
    new_financials,
    columns=financial_columns,
)

# 같은 financial_id가 이미 있으면 기존 행을 제거하고 새 행으로 교체
financials_df = financials_df[
    ~financials_df["financial_id"].isin(new_financials_df["financial_id"])
]

financials_df = pd.concat(
    [financials_df, new_financials_df],
    ignore_index=True,
)

print(f"현재 재무 데이터: {len(financials_df)}건")
display(financials_df)

현재 재무 데이터: 7건


,financial_id,company_id,fiscal_year,period,statement_scope,revenue_krw,operating_profit_krw,source
0,FINANCIAL-AMOREPACIFIC-2025-ANNUAL-OFS,COMPANY-AMOREPACIFIC,2025,연간,개별,2653725793231,212514854678,[https://dart.fss.or.kr/dsaf001/main.do?rcpNo=...
1,FINANCIAL-INNISFREE-2025-ANNUAL-UNKNOWN,COMPANY-INNISFREE,2025,연간,NaN,209848553422,13348911731,"[{'source_type': 'disclosure', 'source_name': ..."
2,FINANCIAL-ESPOIR-2025-ANNUAL-UNKNOWN,COMPANY-ESPOIR,2025,연간,NaN,77210868308,-452624866,"[{'source_type': 'disclosure', 'source_name': ..."
3,FINANCIAL-ETUDE-2025-ANNUAL-UNKNOWN,COMPANY-ETUDE,2025,연간,NaN,115092257274,9971025760,"[{'source_type': 'disclosure', 'source_name': ..."
4,FINANCIAL-AMOSPROFESSIONAL-2025-ANNUAL-UNKNOWN,COMPANY-AMOSPROFESSIONAL,2025,연간,NaN,79136115848,6982117086,"[{'source_type': 'disclosure', 'source_name': ..."
5,FINANCIAL-COSRX-2025-ANNUAL-UNKNOWN,COMPANY-COSRX,2025,연간,NaN,456246005204,114015269459,"[{'source_type': 'disclosure', 'source_name': ..."
6,FINANCIAL-PACIFICTECH-2025-ANNUAL-OFS,COMPANY-PACIFICTECH,2025,연간,개별,1854000000,-922000000,"[{'source_type': 'disclosure', 'source_name': ..."


In [12]:
# 감사보고서에서 회사 자체의 손익계산서를 확인한 기업
individual_companies = [
    "COMPANY-INNISFREE",
    "COMPANY-ESPOIR",
    "COMPANY-ETUDE",
    "COMPANY-AMOSPROFESSIONAL",
    "COMPANY-COSRX",
]

# 최종 재무 데이터 표에 반영
financials_df.loc[
    financials_df["company_id"].isin(individual_companies),
    "statement_scope",
] = "개별"

# 검토용 표에도 동일하게 반영
financial_candidates_df.loc[
    financial_candidates_df["company_id"].isin(individual_companies),
    "statement_scope",
] = "개별"

# 결과 확인
display(
    financials_df[
        [
            "company_id",
            "fiscal_year",
            "period",
            "statement_scope",
            "revenue_krw",
            "operating_profit_krw",
        ]
    ]
)

,company_id,fiscal_year,period,statement_scope,revenue_krw,operating_profit_krw
0,COMPANY-AMOREPACIFIC,2025,연간,개별,2653725793231,212514854678
1,COMPANY-INNISFREE,2025,연간,개별,209848553422,13348911731
2,COMPANY-ESPOIR,2025,연간,개별,77210868308,-452624866
3,COMPANY-ETUDE,2025,연간,개별,115092257274,9971025760
4,COMPANY-AMOSPROFESSIONAL,2025,연간,개별,79136115848,6982117086
5,COMPANY-COSRX,2025,연간,개별,456246005204,114015269459
6,COMPANY-PACIFICTECH,2025,연간,개별,1854000000,-922000000


In [13]:
financials_df.head(10)

,financial_id,company_id,fiscal_year,period,statement_scope,revenue_krw,operating_profit_krw,source
0,FINANCIAL-AMOREPACIFIC-2025-ANNUAL-OFS,COMPANY-AMOREPACIFIC,2025,연간,개별,2653725793231,212514854678,[https://dart.fss.or.kr/dsaf001/main.do?rcpNo=...
1,FINANCIAL-INNISFREE-2025-ANNUAL-UNKNOWN,COMPANY-INNISFREE,2025,연간,개별,209848553422,13348911731,"[{'source_type': 'disclosure', 'source_name': ..."
2,FINANCIAL-ESPOIR-2025-ANNUAL-UNKNOWN,COMPANY-ESPOIR,2025,연간,개별,77210868308,-452624866,"[{'source_type': 'disclosure', 'source_name': ..."
3,FINANCIAL-ETUDE-2025-ANNUAL-UNKNOWN,COMPANY-ETUDE,2025,연간,개별,115092257274,9971025760,"[{'source_type': 'disclosure', 'source_name': ..."
4,FINANCIAL-AMOSPROFESSIONAL-2025-ANNUAL-UNKNOWN,COMPANY-AMOSPROFESSIONAL,2025,연간,개별,79136115848,6982117086,"[{'source_type': 'disclosure', 'source_name': ..."
5,FINANCIAL-COSRX-2025-ANNUAL-UNKNOWN,COMPANY-COSRX,2025,연간,개별,456246005204,114015269459,"[{'source_type': 'disclosure', 'source_name': ..."
6,FINANCIAL-PACIFICTECH-2025-ANNUAL-OFS,COMPANY-PACIFICTECH,2025,연간,개별,1854000000,-922000000,"[{'source_type': 'disclosure', 'source_name': ..."


In [14]:
# 개별 재무제표로 확인된 행의 ID를 정리
updated_ids = financials_df["financial_id"].copy()

individual_mask = financials_df["statement_scope"].eq("개별")

updated_ids.loc[individual_mask] = (
    updated_ids.loc[individual_mask]
    .str.replace(r"-UNKNOWN$", "-OFS", regex=True)
)

# 변경 후 ID가 중복되는지 확인
if updated_ids.duplicated().any():
    raise ValueError("변경 후 중복되는 financial_id가 있습니다. 확인해 주세요.")

# 중복이 없으면 반영
financials_df["financial_id"] = updated_ids

display(financials_df[["financial_id", "company_id", "statement_scope"]])

,financial_id,company_id,statement_scope
0,FINANCIAL-AMOREPACIFIC-2025-ANNUAL-OFS,COMPANY-AMOREPACIFIC,개별
1,FINANCIAL-INNISFREE-2025-ANNUAL-OFS,COMPANY-INNISFREE,개별
2,FINANCIAL-ESPOIR-2025-ANNUAL-OFS,COMPANY-ESPOIR,개별
3,FINANCIAL-ETUDE-2025-ANNUAL-OFS,COMPANY-ETUDE,개별
4,FINANCIAL-AMOSPROFESSIONAL-2025-ANNUAL-OFS,COMPANY-AMOSPROFESSIONAL,개별
5,FINANCIAL-COSRX-2025-ANNUAL-OFS,COMPANY-COSRX,개별
6,FINANCIAL-PACIFICTECH-2025-ANNUAL-OFS,COMPANY-PACIFICTECH,개별


In [15]:
financials_df.head(10)

,financial_id,company_id,fiscal_year,period,statement_scope,revenue_krw,operating_profit_krw,source
0,FINANCIAL-AMOREPACIFIC-2025-ANNUAL-OFS,COMPANY-AMOREPACIFIC,2025,연간,개별,2653725793231,212514854678,[https://dart.fss.or.kr/dsaf001/main.do?rcpNo=...
1,FINANCIAL-INNISFREE-2025-ANNUAL-OFS,COMPANY-INNISFREE,2025,연간,개별,209848553422,13348911731,"[{'source_type': 'disclosure', 'source_name': ..."
2,FINANCIAL-ESPOIR-2025-ANNUAL-OFS,COMPANY-ESPOIR,2025,연간,개별,77210868308,-452624866,"[{'source_type': 'disclosure', 'source_name': ..."
3,FINANCIAL-ETUDE-2025-ANNUAL-OFS,COMPANY-ETUDE,2025,연간,개별,115092257274,9971025760,"[{'source_type': 'disclosure', 'source_name': ..."
4,FINANCIAL-AMOSPROFESSIONAL-2025-ANNUAL-OFS,COMPANY-AMOSPROFESSIONAL,2025,연간,개별,79136115848,6982117086,"[{'source_type': 'disclosure', 'source_name': ..."
5,FINANCIAL-COSRX-2025-ANNUAL-OFS,COMPANY-COSRX,2025,연간,개별,456246005204,114015269459,"[{'source_type': 'disclosure', 'source_name': ..."
6,FINANCIAL-PACIFICTECH-2025-ANNUAL-OFS,COMPANY-PACIFICTECH,2025,연간,개별,1854000000,-922000000,"[{'source_type': 'disclosure', 'source_name': ..."


In [16]:
# 아모레퍼시픽 재무 데이터 행 찾기
amore_mask = financials_df["company_id"] == "COMPANY-AMOREPACIFIC"

if amore_mask.sum() != 1:
    raise ValueError("아모레퍼시픽 재무 데이터를 하나로 특정할 수 없습니다.")

# 기존에 저장된 공시 URL 가져오기
amore_index = financials_df.index[amore_mask][0]
existing_source = financials_df.at[amore_index, "source"]

# 이미 객체 배열로 바뀌었다면 그대로 두고, URL 배열이면 변환
if (
    isinstance(existing_source, list)
    and len(existing_source) == 1
    and isinstance(existing_source[0], str)
):
    financials_df.at[amore_index, "source"] = [
        {
            "source_type": "disclosure",
            "source_name": "아모레퍼시픽 사업보고서",
            "original_url": existing_source[0],
            "collected_at": None,
            "as_of_date": "2025-12-31",
            "evidence_location": "개별 포괄손익계산서",
        }
    ]

# 모든 행이 출처 객체 배열인지 확인
source_check = financials_df["source"].apply(
    lambda sources: (
        isinstance(sources, list)
        and len(sources) > 0
        and all(isinstance(source, dict) for source in sources)
    )
)

print("출처 객체 배열 형식:", "모두 확인됨" if source_check.all() else "확인 필요")
display(financials_df[["company_id", "source"]])

출처 객체 배열 형식: 모두 확인됨


,company_id,source
0,COMPANY-AMOREPACIFIC,"[{'source_type': 'disclosure', 'source_name': ..."
1,COMPANY-INNISFREE,"[{'source_type': 'disclosure', 'source_name': ..."
2,COMPANY-ESPOIR,"[{'source_type': 'disclosure', 'source_name': ..."
3,COMPANY-ETUDE,"[{'source_type': 'disclosure', 'source_name': ..."
4,COMPANY-AMOSPROFESSIONAL,"[{'source_type': 'disclosure', 'source_name': ..."
5,COMPANY-COSRX,"[{'source_type': 'disclosure', 'source_name': ..."
6,COMPANY-PACIFICTECH,"[{'source_type': 'disclosure', 'source_name': ..."


In [17]:
financials_df.head(10)

,financial_id,company_id,fiscal_year,period,statement_scope,revenue_krw,operating_profit_krw,source
0,FINANCIAL-AMOREPACIFIC-2025-ANNUAL-OFS,COMPANY-AMOREPACIFIC,2025,연간,개별,2653725793231,212514854678,"[{'source_type': 'disclosure', 'source_name': ..."
1,FINANCIAL-INNISFREE-2025-ANNUAL-OFS,COMPANY-INNISFREE,2025,연간,개별,209848553422,13348911731,"[{'source_type': 'disclosure', 'source_name': ..."
2,FINANCIAL-ESPOIR-2025-ANNUAL-OFS,COMPANY-ESPOIR,2025,연간,개별,77210868308,-452624866,"[{'source_type': 'disclosure', 'source_name': ..."
3,FINANCIAL-ETUDE-2025-ANNUAL-OFS,COMPANY-ETUDE,2025,연간,개별,115092257274,9971025760,"[{'source_type': 'disclosure', 'source_name': ..."
4,FINANCIAL-AMOSPROFESSIONAL-2025-ANNUAL-OFS,COMPANY-AMOSPROFESSIONAL,2025,연간,개별,79136115848,6982117086,"[{'source_type': 'disclosure', 'source_name': ..."
5,FINANCIAL-COSRX-2025-ANNUAL-OFS,COMPANY-COSRX,2025,연간,개별,456246005204,114015269459,"[{'source_type': 'disclosure', 'source_name': ..."
6,FINANCIAL-PACIFICTECH-2025-ANNUAL-OFS,COMPANY-PACIFICTECH,2025,연간,개별,1854000000,-922000000,"[{'source_type': 'disclosure', 'source_name': ..."


In [18]:
from pathlib import Path
import json

# 저장할 파일 경로
output_path = Path(
    "/Users/anna/bulsae/data/collected/anna/financials.jsonl"
)

# 저장할 컬럼과 순서
financial_columns = [
    "financial_id",
    "company_id",
    "fiscal_year",
    "period",
    "statement_scope",
    "revenue_krw",
    "operating_profit_krw",
    "source",
]

# 필요한 컬럼만 선택
output_df = financials_df[financial_columns].copy()

# 저장 전 기본 확인
if output_df.empty:
    raise ValueError("저장할 재무 데이터가 없습니다.")

if output_df["financial_id"].isna().any():
    raise ValueError("financial_id가 비어 있는 행이 있습니다.")

if output_df["financial_id"].duplicated().any():
    raise ValueError("중복된 financial_id가 있습니다.")

source_valid = output_df["source"].apply(
    lambda sources: (
        isinstance(sources, list)
        and len(sources) > 0
        and all(isinstance(source, dict) for source in sources)
    )
)

if not source_valid.all():
    raise ValueError("source가 출처 객체 배열이 아닌 행이 있습니다.")

# JSONL로 변환하고, 각 줄이 JSON 객체인지 확인
jsonl_text = output_df.to_json(
    orient="records",
    lines=True,
    force_ascii=False,
)

records = [json.loads(line) for line in jsonl_text.splitlines() if line.strip()]

if len(records) != len(output_df):
    raise ValueError("저장할 데이터의 행 수가 일치하지 않습니다.")

# 파일 저장: 기존 파일 내용을 새 데이터로 교체
output_path.parent.mkdir(parents=True, exist_ok=True)
output_path.write_text(jsonl_text, encoding="utf-8")

print(f"저장 완료: {output_path}")
print(f"저장된 재무 데이터: {len(records)}건")

저장 완료: /Users/anna/bulsae/data/collected/anna/financials.jsonl
저장된 재무 데이터: 7건


In [19]:
# financial_contents.jsonl 스키마
financial_content_columns = [
    "content_id",
    "company_id",
    "fiscal_year",
    "content",
    "source",
    "verification_status",
]

# 재무 이슈를 담을 빈 표
financial_contents_df = pd.DataFrame(
    columns=financial_content_columns
)

print(f"현재 재무 이슈: {len(financial_contents_df)}건")
display(financial_contents_df)

현재 재무 이슈: 0건


,content_id,company_id,fiscal_year,content,source,verification_status


In [23]:
# 표의 긴 문자열을 생략하지 않고 표시
pd.set_option("display.max_colwidth", None)

# 제목과 요약만 먼저 확인
display(news_df[["title", "description"]])

,title,description
0,"[주목 e-맞수] ""영업익 늘었지만..."" LG생건 vs 아모레퍼시픽…""中·음료...","이 같은 실망감에 실적 발표 이튿날인 11월 4일 유가증권시장에서 LG생건 주가는 전 거래일 대비 11.83%(3만 1... 국내 사업 영업이익이 846억 원으로 42% 급증한 데다, 미주와 일본에서 에스트라·코스알엑스(COSRX)·설화수..."
1,K-뷰티 성장엔진 ‘다각화’…브랜드·제조사 실적 ‘동반 질주’,◇ 에이피알·아모레퍼시픽 호실적…한국콜마·코스맥스도 '성장 가속' 5일 업계에 따르면 에이피알은... 일본에서는 에스트라와 코스알엑스 매출이 각각 두 배 이상 뛰었다. 인도 매출도 60% 이상 늘었다....
2,"[K-뷰티 신화를 쓰다②] K뷰티, '난장이'가 쏘아 올렸다…한국콜마·올리...",생산실적도 3328억 원에서 1661억 원으로 감소했다. 코스알엑스 역시 같은 기간 6위에서 20위로 하락하는 등 신흥 브랜드 기업들 사이에서도 순위 변동이 나타나고 있다. 다만 생산실적은 국내에서 생산된 화장품을...
3,미국·유럽서 웃고 중국서 주춤…K뷰티 성장 열쇠는 해외 시장,"일본에선 코스알엑스가 현지 매장 약 2900곳에 새로 입점하며 매출이 전년 동기 대비 두 배 이상 늘었다.... 려와 헤라의 판매는 늘었지만, 설화수의 오프라인 유통 채널 효율화에 따른 매출 감소 등이 실적에 영향을..."
4,기능성 키우고 유통망 넓히고···새 판 짜는 K뷰티,"올 3분기 국내 뷰티기업들의 실적은 희비가 엇갈렸다. 아모레퍼시픽그룹의 올해 3분기 연결 기준 매출이... 일본에서는 에스트라와 코스알엑스가 각각 100% 이상, 헤라가 90% 이상 신장했다. 코스알엑스는 현지 약 2900개..."
5,"'날개 단' 아모레, '뛰는' LG생건…수익성 개선에도 '희비'",국내 화장품 대표 기업 아모레퍼시픽과 LG생활건강이 올해 3분기 서로 다른 실적 흐름을 보였다.... 일본에서는 에스트라와 코스알엑스 매출이 각각 100% 성장했으며 헤라는 90% 이상 늘었다. 인도에서는 라네즈를 비롯한...
6,"[리더&CEO] 서경배 뉴뷰티 1년…아모레퍼시픽, 화장품 속 과학을 팔다",이 철학이 최근 실적 반등의 밑바탕이 되고 있다. 아모레퍼시픽 그룹의 올해 1~3분기 누적 영업이익은 3907억... 코스알엑스는 새 소재를 빠르게 제품으로 만드는 방식으로 성장하고 있다. AI 분자 모델링으로 찾아낸...
7,"코스알엑스, 호주 케이마트·타겟 입점… 현지 판매처 2000곳으로",스킨케어 브랜드 코스알엑스(COSRX)가 이달 중순부터 호주 할인 체인 케이마트(Kmart)와 타겟(Target)에... 회사 관계자는 “한국과 미국에서 판매 실적을 쌓은 펩타이드 라인을 호주에도 선보이게 됐다”며 “현지...
8,"[비즈 인사이트] 화장품 빅4, 3분기 판도 변화…에이피알 영업이익, 아모...",코스알엑스의 판매가 늘었다. 인도 매출도 60% 이상 증가했다. 국내 사업의 영업이익 증가율은 해외보다 28%포인트 높았다. LG생활건강은 북미 자회사 더 에이본 컴퍼니를 매각하면서 9월부터 해당 법인의 실적을...
9,"DB증권 ""아모레퍼시픽, 예상보다 좋은 실적…목표가 유지""","4분기에는 국가별 실적 방향성이 나뉠 것으로 관측된다. 코스알엑스와 라네즈의 EMEA 부진으로 자체 브랜드의 4분기 매출은 두 자릿수 하락이 예상되나, 중국의 브랜드 및 채널 구조조정이 마무리 국면에 접어들면서..."


In [24]:
import os
import html
import re

import pandas as pd
import requests
from dotenv import load_dotenv

# .env 파일에서 API 키 불러오기
load_dotenv("/Users/anna/bulsae/.env", override=True)

naver_client_id = os.getenv("NAVER_CLIENT_ID")
naver_client_secret = os.getenv("NAVER_CLIENT_SECRET")

if not naver_client_id or not naver_client_secret:
    raise ValueError(".env에서 네이버 API 키를 찾을 수 없습니다.")


def clean_news_text(text):
    """검색 결과에 포함된 HTML 태그와 특수문자를 정리합니다."""
    text = re.sub(r"<[^>]+>", "", text or "")
    return html.unescape(text).strip()


# 코스알엑스 실적 기사 검색
url = "https://naverapihub.apigw.ntruss.com/search/v1/news"

headers = {
    "X-NCP-APIGW-API-KEY-ID": naver_client_id,
    "X-NCP-APIGW-API-KEY": naver_client_secret,
}

params = {
    "query": "코스알엑스 실적",
    "display": 10,
    "sort": "date",
}

response = requests.get(
    url,
    headers=headers,
    params=params,
    timeout=30,
)

response.raise_for_status()
data = response.json()

# 기사 후보를 표로 정리
news_df = pd.DataFrame([
    {
        "title": clean_news_text(item.get("title")),
        "description": clean_news_text(item.get("description")),
        "original_url": item.get("originallink"),
        "naver_url": item.get("link"),
        "published_at": item.get("pubDate"),
    }
    for item in data.get("items", [])
])

print(f"조회된 기사: {len(news_df)}건")
display(news_df)

조회된 기사: 10건


,title,description,original_url,naver_url,published_at
0,"[주목 e-맞수] ""영업익 늘었지만..."" LG생건 vs 아모레퍼시픽…""中·음료...","이 같은 실망감에 실적 발표 이튿날인 11월 4일 유가증권시장에서 LG생건 주가는 전 거래일 대비 11.83%(3만 1... 국내 사업 영업이익이 846억 원으로 42% 급증한 데다, 미주와 일본에서 에스트라·코스알엑스(COSRX)·설화수...",http://www.economytalk.kr/news/articleView.html?idxno=425107,http://www.economytalk.kr/news/articleView.html?idxno=425107,"Thu, 05 Nov 2026 10:46:00 +0900"
1,K-뷰티 성장엔진 ‘다각화’…브랜드·제조사 실적 ‘동반 질주’,◇ 에이피알·아모레퍼시픽 호실적…한국콜마·코스맥스도 '성장 가속' 5일 업계에 따르면 에이피알은... 일본에서는 에스트라와 코스알엑스 매출이 각각 두 배 이상 뛰었다. 인도 매출도 60% 이상 늘었다....,https://www.ekn.kr/web/view.php?key=20261105021475369,https://www.ekn.kr/web/view.php?key=20261105021475369,"Thu, 05 Nov 2026 08:10:00 +0900"
2,"[K-뷰티 신화를 쓰다②] K뷰티, '난장이'가 쏘아 올렸다…한국콜마·올리...",생산실적도 3328억 원에서 1661억 원으로 감소했다. 코스알엑스 역시 같은 기간 6위에서 20위로 하락하는 등 신흥 브랜드 기업들 사이에서도 순위 변동이 나타나고 있다. 다만 생산실적은 국내에서 생산된 화장품을...,http://www.consumernews.co.kr/news/articleView.html?idxno=766431,http://www.consumernews.co.kr/news/articleView.html?idxno=766431,"Thu, 05 Nov 2026 06:16:00 +0900"
3,미국·유럽서 웃고 중국서 주춤…K뷰티 성장 열쇠는 해외 시장,"일본에선 코스알엑스가 현지 매장 약 2900곳에 새로 입점하며 매출이 전년 동기 대비 두 배 이상 늘었다.... 려와 헤라의 판매는 늘었지만, 설화수의 오프라인 유통 채널 효율화에 따른 매출 감소 등이 실적에 영향을...",https://www.sisaon.co.kr/news/articleView.html?idxno=205796,https://www.sisaon.co.kr/news/articleView.html?idxno=205796,"Wed, 04 Nov 2026 16:50:00 +0900"
4,기능성 키우고 유통망 넓히고···새 판 짜는 K뷰티,"올 3분기 국내 뷰티기업들의 실적은 희비가 엇갈렸다. 아모레퍼시픽그룹의 올해 3분기 연결 기준 매출이... 일본에서는 에스트라와 코스알엑스가 각각 100% 이상, 헤라가 90% 이상 신장했다. 코스알엑스는 현지 약 2900개...",http://www.enewstoday.co.kr/news/articleView.html?idxno=2488306,http://www.enewstoday.co.kr/news/articleView.html?idxno=2488306,"Wed, 04 Nov 2026 14:33:00 +0900"
5,"'날개 단' 아모레, '뛰는' LG생건…수익성 개선에도 '희비'",국내 화장품 대표 기업 아모레퍼시픽과 LG생활건강이 올해 3분기 서로 다른 실적 흐름을 보였다.... 일본에서는 에스트라와 코스알엑스 매출이 각각 100% 성장했으며 헤라는 90% 이상 늘었다. 인도에서는 라네즈를 비롯한...,https://news.bizwatch.co.kr/article/consumer/2026/11/04/0018,https://n.news.naver.com/mnews/article/648/0000052276?sid=101,"Wed, 04 Nov 2026 14:28:00 +0900"
6,"[리더&CEO] 서경배 뉴뷰티 1년…아모레퍼시픽, 화장품 속 과학을 팔다",이 철학이 최근 실적 반등의 밑바탕이 되고 있다. 아모레퍼시픽 그룹의 올해 1~3분기 누적 영업이익은 3907억... 코스알엑스는 새 소재를 빠르게 제품으로 만드는 방식으로 성장하고 있다. AI 분자 모델링으로 찾아낸...,https://www.cnbizm.com/news/articleView.html?idxno=320329,https://www.cnbizm.com/news/articleView.html?idxno=320329,"Wed, 04 Nov 2026 14:20:00 +0900"
7,"코스알엑스, 호주 케이마트·타겟 입점… 현지 판매처 2000곳으로",스킨케어 브랜드 코스알엑스(COSRX)가 이달 중순부터 호주 할인 체인 케이마트(Kmart)와 타겟(Target)에... 회사 관계자는 “한국과 미국에서 판매 실적을 쌓은 펩타이드 라인을 호주에도 선보이게 됐다”며 “현지...,https://www.donga.com/news/Economy/article/all/20261104/134963787/1,https://n.news.naver.com/mnews/article/020/0003759987?sid=101,"Wed, 04 Nov 2026 13:19:00 +0900"
8,"[비즈 인사이트] 화장품 빅4, 3분기 판도 변화…에이피알 영업이익, 아모...",코스알엑스의 판매가 늘었다. 인도 매출도 60% 이상 증가했다. 국내 사업의 영업이익 증가율은 해외보다 28%포인트 높았다. LG생활건강은 북미 자회사 더 에이본 컴퍼니를 매각하면서 9월부터 해당 법인의 실적을...,https://www.seoultimes.news/news/article.html?no=2000101733,https://www.seoultimes.news/news/article.html?no=2000101733,"Wed, 04 Nov 2026 09:00:00 +0900"
9,"DB증권 ""아모레퍼시픽, 예상보다 좋은 실적…목표가 유지""","4분기에는 국가별 실적 방향성이 나뉠 것으로 관측된다. 코스알엑스와 라네즈의 EMEA 부진으로 자체 브랜드의 4분기 매출은 두 자릿수 하락이 예상되나, 중국의 브랜드 및 채널 구조조정이 마무리 국면에 접어들면서...",https://www.jasm.co.kr/news/view/324030/?sc=Naver,https://www.jasm.co.kr/news/view/324030/?sc=Naver,"Wed, 04 Nov 2026 08:22:00 +0900"


In [31]:
news_df['original_url']

0           http://www.economytalk.kr/news/articleView.html?idxno=425107
1                  https://www.ekn.kr/web/view.php?key=20261105021475369
2       http://www.consumernews.co.kr/news/articleView.html?idxno=766431
3            https://www.sisaon.co.kr/news/articleView.html?idxno=205796
4        http://www.enewstoday.co.kr/news/articleView.html?idxno=2488306
5           https://news.bizwatch.co.kr/article/consumer/2026/11/04/0018
6              https://www.cnbizm.com/news/articleView.html?idxno=320329
7    https://www.donga.com/news/Economy/article/all/20261104/134963787/1
8            https://www.seoultimes.news/news/article.html?no=2000101733
9                      https://www.jasm.co.kr/news/view/324030/?sc=Naver
Name: original_url, dtype: str

In [25]:
for index, row in news_df.iterrows():
    print(f"\n[{index}] {row['title']}")
    print(f"요약: {row['description']}")
    print(f"원문: {row['original_url']}")
    print("-" * 80)


[0] [주목 e-맞수] "영업익 늘었지만..." LG생건 vs 아모레퍼시픽…"中·음료...
요약: 이 같은 실망감에 실적 발표 이튿날인 11월 4일 유가증권시장에서 LG생건 주가는 전 거래일 대비 11.83%(3만 1... 국내 사업 영업이익이 846억 원으로 42% 급증한 데다, 미주와 일본에서 에스트라·코스알엑스(COSRX)·설화수...
원문: http://www.economytalk.kr/news/articleView.html?idxno=425107
--------------------------------------------------------------------------------

[1] K-뷰티 성장엔진 ‘다각화’…브랜드·제조사 실적 ‘동반 질주’
요약: ◇ 에이피알·아모레퍼시픽 호실적…한국콜마·코스맥스도 '성장 가속' 5일 업계에 따르면 에이피알은... 일본에서는 에스트라와 코스알엑스 매출이 각각 두 배 이상 뛰었다. 인도 매출도 60% 이상 늘었다....
원문: https://www.ekn.kr/web/view.php?key=20261105021475369
--------------------------------------------------------------------------------

[2] [K-뷰티 신화를 쓰다②] K뷰티, '난장이'가 쏘아 올렸다…한국콜마·올리...
요약: 생산실적도 3328억 원에서 1661억 원으로 감소했다. 코스알엑스 역시 같은 기간 6위에서 20위로 하락하는 등 신흥 브랜드 기업들 사이에서도 순위 변동이 나타나고 있다. 다만 생산실적은 국내에서 생산된 화장품을...
원문: http://www.consumernews.co.kr/news/articleView.html?idxno=766431
--------------------------------------------------------------------------------

[3] 미국·유럽서 웃고 중국서 주춤…K뷰티 성장 열쇠는 해

In [32]:
import requests
from bs4 import BeautifulSoup

# 기사 URL
article_url = "https://www.economytalk.kr/news/articleView.html?idxno=425107"

# 기사 페이지 요청
response = requests.get(article_url, timeout=30)
response.raise_for_status()

# HTML을 BeautifulSoup으로 읽기
soup = BeautifulSoup(response.content, "html.parser")

# 기사 제목
title_tag = soup.select_one("h1")
title = title_tag.get_text(" ", strip=True) if title_tag else None

# 기사 본문
article_tag = soup.select_one("#article-view-content-div")

if article_tag is None:
    raise ValueError(
        "기사 본문을 찾지 못했습니다. "
        "페이지 구조가 바뀌었거나 요청이 차단됐는지 확인해 주세요."
    )

# 본문 안의 스크립트와 스타일은 제거
for tag in article_tag.select("script, style"):
    tag.decompose()

# 문단과 표의 텍스트를 줄 단위로 추출
article_text = article_tag.get_text(separator="\n", strip=True)

print("제목:", title)
print("원문:", article_url)
print("\n본문:\n")
print(article_text)

제목: [주목 e-맞수] "영업익 늘었지만..." LG생건 vs 아모레퍼시픽…"中·음료 부진은 부담, Q4 턴어라운드?"
원문: https://www.economytalk.kr/news/articleView.html?idxno=425107

본문:

[김지현 기자 @이코노미톡뉴스] LG생활건강(이하 LG생건)이 2026년 3분기 뷰티 부문의 적자 폭을 줄이며 전년 동기 대비 50%가 넘는 영업이익 증가율을 기록했으나, 시장 기대치를 크게 밑도는 ‘어닝 쇼크’ 성적표를 받아들었다. 만성 적자 늪에 빠졌던 북미 자회사 에이본(The Avon Company)을 매각하며 재무 불확실성을 도려내는 결단을 내렸지만, 중국 시장의 구조적 침체와 캐시카우(현금창출원)였던 음료 부문의 수익성 악화가 겹치며 성장 동력이 둔화됐다는 평가다. 특히 같은 기간 아모레퍼시픽이 두 자릿수 영업이익률을 회복하고, 후발주자인 에이피알(APR)이 화장품 매출과 전사 영업이익에서 모두 LG생건을 추월하면서 국내 뷰티 업계의 지형도 변화가 한층 뚜렷해지고 있다.
LG생건: 영업익 증가는 기저효과…"화장품 적자 지속·음료 수익성↓"
금융감독원 전자공시시스템에 따르면 LG생건의 2026년 3분기 연결 기준 매출액은 1조 5,774억 원으로 전년 동기 대비 0.2% 감소했고, 영업이익은 704억 원으로 52.4% 증가했다. 표면적으로는 영업이익이 큰 폭으로 반등한 것처럼 보이지만, 이는 지난해 3분기 기저효과에 따른 결과로 증권가 컨센서스(전망치 평균 약 1,013억 원)를 30.5% 밑돈 수치다.
LG생활건강은 2026년 3분기 영업이익이 전년 동기 대비 52.4% 증가한 704억 원을 기록하며 외형상 실적 개선에 성공했지만, 시장 기대치에는 크게 못 미치는 성적표를 받아들었다. 매출은 1조 5,774억 원으로 소폭 감소했고, 영업이익은 증권가 컨센서스를 약 30% 하회했다. [이미지=e톡뉴스 AI]
직전 분기(2026년 2분기)와 비교하면 수익성 둔화는 더욱 두드러진다. 전분기 대비 매출은 4.8

In [ ]:
import re
import time

import pandas as pd
import requests
from bs4 import BeautifulSoup
from markdownify import markdownify as html_to_markdown
from IPython.display import display, Markdown


# 언론사에서 자주 사용하는 기사 본문 선택자
# 앞에서부터 순서대로 확인합니다.
ARTICLE_SELECTORS = [
    "#article-view-content-div",  # 이코노미톡뉴스 등
    "#article-view-content",
    "#articleBody",
    "#article-body",
    "#articleBodyContents",
    "#newsContent",
    "#news-content",
    "#newsct_article",
    ".article-view-content",
    ".article-body",
    ".articleBody",
    ".article-content",
    ".news-body",
    ".news_view",
    "article",
]


def collect_article(url, timeout=20):
    """
    기사 URL 하나를 요청해 제목과 본문을 마크다운으로 변환합니다.

    실패하면 오류 정보를 담아 반환하므로
    다른 기사 수집은 계속 진행할 수 있습니다.
    """
    result = {
        "url": url,
        "final_url": None,
        "title": None,
        "markdown": None,
        "selector": None,
        "status": "실패",
        "error": None,
    }

    try:
        response = requests.get(
            url,
            headers={
                "User-Agent": "Mozilla/5.0 (compatible; ResearchArticleReader/1.0)",
            },
            timeout=timeout,
        )
        response.raise_for_status()

        result["final_url"] = response.url

        # 페이지가 지정한 인코딩을 우선 사용하고,
        # 인코딩 정보가 없으면 추정값을 사용합니다.
        if not response.encoding or response.encoding.lower() == "iso-8859-1":
            response.encoding = response.apparent_encoding

        soup = BeautifulSoup(response.text, "html.parser")

        # 기사 제목
        title_tag = (
            soup.select_one("meta[property='og:title']")
            or soup.select_one("h1")
            or soup.select_one("title")
        )

        if title_tag:
            if title_tag.name == "meta":
                result["title"] = title_tag.get("content", "").strip()
            else:
                result["title"] = title_tag.get_text(" ", strip=True)

        # 본문으로 보이는 영역 찾기
        article_tag = None

        for selector in ARTICLE_SELECTORS:
            candidate = soup.select_one(selector)

            if candidate and len(candidate.get_text(" ", strip=True)) >= 100:
                article_tag = candidate
                result["selector"] = selector
                break

        if article_tag is None:
            result["error"] = "본문 영역을 찾지 못했습니다."
            return result

        # 광고·스크립트 등은 본문 변환에서 제외
        for tag in article_tag.select(
            "script, style, iframe, form, button, "
            ".advertisement, .advert, .ad"
        ):
            tag.decompose()

        # HTML을 Markdown으로 변환
        # 표가 있는 경우 가능한 범위에서 Markdown 표로 변환합니다.
        markdown_text = html_to_markdown(
            str(article_tag),
            heading_style="ATX",
            table_infer_header=True,
        ).strip()

        # 빈 줄이 지나치게 많으면 정리
        markdown_text = re.sub(r"\n{3,}", "\n\n", markdown_text)

        if not markdown_text:
            result["error"] = "본문 영역은 찾았지만 내용이 비어 있습니다."
            return result

        result["markdown"] = markdown_text
        result["status"] = "확인 필요"

    except requests.exceptions.Timeout:
        result["error"] = "요청 시간이 초과되었습니다."

    except requests.exceptions.HTTPError as error:
        status_code = error.response.status_code if error.response is not None else None
        result["error"] = f"HTTP 오류: {status_code}"

    except requests.exceptions.RequestException as error:
        result["error"] = f"페이지 요청 실패: {type(error).__name__}"

    except Exception as error:
        result["error"] = f"변환 실패: {type(error).__name__}: {error}"

    return result

In [33]:
import re
import time

import pandas as pd
import requests
from bs4 import BeautifulSoup
from markdownify import markdownify as html_to_markdown
from IPython.display import display, Markdown


# 언론사에서 자주 사용하는 기사 본문 선택자
# 앞에서부터 순서대로 확인합니다.
ARTICLE_SELECTORS = [
    "#article-view-content-div",  # 이코노미톡뉴스 등
    "#article-view-content",
    "#articleBody",
    "#article-body",
    "#articleBodyContents",
    "#newsContent",
    "#news-content",
    "#newsct_article",
    ".article-view-content",
    ".article-body",
    ".articleBody",
    ".article-content",
    ".news-body",
    ".news_view",
    "article",
]


def collect_article(url, timeout=20):
    """
    기사 URL 하나를 요청해 제목과 본문을 마크다운으로 변환합니다.

    실패하면 오류 정보를 담아 반환하므로
    다른 기사 수집은 계속 진행할 수 있습니다.
    """
    result = {
        "url": url,
        "final_url": None,
        "title": None,
        "markdown": None,
        "selector": None,
        "status": "실패",
        "error": None,
    }

    try:
        response = requests.get(
            url,
            headers={
                "User-Agent": "Mozilla/5.0 (compatible; ResearchArticleReader/1.0)",
            },
            timeout=timeout,
        )
        response.raise_for_status()

        result["final_url"] = response.url

        # 페이지가 지정한 인코딩을 우선 사용하고,
        # 인코딩 정보가 없으면 추정값을 사용합니다.
        if not response.encoding or response.encoding.lower() == "iso-8859-1":
            response.encoding = response.apparent_encoding

        soup = BeautifulSoup(response.text, "html.parser")

        # 기사 제목
        title_tag = (
            soup.select_one("meta[property='og:title']")
            or soup.select_one("h1")
            or soup.select_one("title")
        )

        if title_tag:
            if title_tag.name == "meta":
                result["title"] = title_tag.get("content", "").strip()
            else:
                result["title"] = title_tag.get_text(" ", strip=True)

        # 본문으로 보이는 영역 찾기
        article_tag = None

        for selector in ARTICLE_SELECTORS:
            candidate = soup.select_one(selector)

            if candidate and len(candidate.get_text(" ", strip=True)) >= 100:
                article_tag = candidate
                result["selector"] = selector
                break

        if article_tag is None:
            result["error"] = "본문 영역을 찾지 못했습니다."
            return result

        # 광고·스크립트 등은 본문 변환에서 제외
        for tag in article_tag.select(
            "script, style, iframe, form, button, "
            ".advertisement, .advert, .ad"
        ):
            tag.decompose()

        # HTML을 Markdown으로 변환
        # 표가 있는 경우 가능한 범위에서 Markdown 표로 변환합니다.
        markdown_text = html_to_markdown(
            str(article_tag),
            heading_style="ATX",
            table_infer_header=True,
        ).strip()

        # 빈 줄이 지나치게 많으면 정리
        markdown_text = re.sub(r"\n{3,}", "\n\n", markdown_text)

        if not markdown_text:
            result["error"] = "본문 영역은 찾았지만 내용이 비어 있습니다."
            return result

        result["markdown"] = markdown_text
        result["status"] = "확인 필요"

    except requests.exceptions.Timeout:
        result["error"] = "요청 시간이 초과되었습니다."

    except requests.exceptions.HTTPError as error:
        status_code = error.response.status_code if error.response is not None else None
        result["error"] = f"HTTP 오류: {status_code}"

    except requests.exceptions.RequestException as error:
        result["error"] = f"페이지 요청 실패: {type(error).__name__}"

    except Exception as error:
        result["error"] = f"변환 실패: {type(error).__name__}: {error}"

    return result

In [34]:
# 빈 URL과 중복 URL 제외
article_urls = (
    news_df["original_url"]
    .dropna()
    .astype(str)
    .str.strip()
)

article_urls = article_urls[article_urls.str.startswith(("http://", "https://"))]
article_urls = article_urls.drop_duplicates().tolist()

results = []

for index, url in enumerate(article_urls, start=1):
    print(f"[{index}/{len(article_urls)}] 수집 중: {url}")

    result = collect_article(url)
    results.append(result)

    if result["error"]:
        print(f"  → 실패: {result['error']}")
    else:
        print(f"  → 본문 후보 발견: {result['selector']}")

    # 사이트에 연속 요청을 보내지 않도록 잠시 대기
    if index < len(article_urls):
        time.sleep(1)

articles_df = pd.DataFrame(results)

print("\n수집 결과")
display(
    articles_df[["title", "status", "selector", "error", "url"]]
)

[1/10] 수집 중: http://www.economytalk.kr/news/articleView.html?idxno=425107
  → 본문 후보 발견: #article-view-content-div
[2/10] 수집 중: https://www.ekn.kr/web/view.php?key=20261105021475369
  → 실패: 본문 영역을 찾지 못했습니다.
[3/10] 수집 중: http://www.consumernews.co.kr/news/articleView.html?idxno=766431
  → 본문 후보 발견: #article-view-content-div
[4/10] 수집 중: https://www.sisaon.co.kr/news/articleView.html?idxno=205796
  → 본문 후보 발견: #article-view-content-div
[5/10] 수집 중: http://www.enewstoday.co.kr/news/articleView.html?idxno=2488306
  → 본문 후보 발견: #article-view-content-div
[6/10] 수집 중: https://news.bizwatch.co.kr/article/consumer/2026/11/04/0018
  → 본문 후보 발견: article
[7/10] 수집 중: https://www.cnbizm.com/news/articleView.html?idxno=320329
  → 본문 후보 발견: #article-view-content-div
[8/10] 수집 중: https://www.donga.com/news/Economy/article/all/20261104/134963787/1
  → 본문 후보 발견: .news_view
[9/10] 수집 중: https://www.seoultimes.news/news/article.html?no=2000101733
  → 실패: 본문 영역을 찾지 못했습니다.
[10/10] 수집 중: https://www.jasm.co.k

,title,status,selector,error,url
0,"[주목 e-맞수] ""영업익 늘었지만..."" LG생건 vs 아모레퍼시픽…""中·음료 부진은 부담, Q4 턴어라운드?"" - 이코노미톡뉴스",확인 필요,#article-view-content-div,NaN,http://www.economytalk.kr/news/articleView.html?idxno=425107
1,K-뷰티 성장엔진 ‘다각화’…브랜드·제조사 실적 ‘동반 질주’,실패,NaN,본문 영역을 찾지 못했습니다.,https://www.ekn.kr/web/view.php?key=20261105021475369
2,"[K-뷰티 신화를 쓰다②] K뷰티, '난장이'가 쏘아 올렸다…한국콜마·올리브영, 중기 수출 생태계 떠받쳐",확인 필요,#article-view-content-div,NaN,http://www.consumernews.co.kr/news/articleView.html?idxno=766431
3,미국·유럽서 웃고 중국서 주춤…K뷰티 성장 열쇠는 해외 시장 - 시사오늘(시사ON),확인 필요,#article-view-content-div,NaN,https://www.sisaon.co.kr/news/articleView.html?idxno=205796
4,기능성 키우고 유통망 넓히고···새 판 짜는 K뷰티,확인 필요,#article-view-content-div,NaN,http://www.enewstoday.co.kr/news/articleView.html?idxno=2488306
5,"'날개 단' 아모레, '뛰는' LG생건…수익성 개선에도 '희비'",확인 필요,article,NaN,https://news.bizwatch.co.kr/article/consumer/2026/11/04/0018
6,"[리더&CEO] 서경배 뉴뷰티 1년…아모레퍼시픽, 화장품 속 과학을 팔다",확인 필요,#article-view-content-div,NaN,https://www.cnbizm.com/news/articleView.html?idxno=320329
7,"코스알엑스, 호주 케이마트·타겟 입점… 현지 판매처 2000곳으로",확인 필요,.news_view,NaN,https://www.donga.com/news/Economy/article/all/20261104/134963787/1
8,"[서울타임즈뉴스] [비즈 인사이트] 화장품 빅4, 3분기 판도 변화…에이피알 영업이익, 아모레·LG생건 추월",실패,NaN,본문 영역을 찾지 못했습니다.,https://www.seoultimes.news/news/article.html?no=2000101733
9,"DB증권 ""아모레퍼시픽, 예상보다 좋은 실적…목표가 유지""",확인 필요,.article-content,NaN,https://www.jasm.co.kr/news/view/324030/?sc=Naver


In [35]:
from IPython.display import display, Markdown

# 0번 기사의 마크다운 본문 보기
display(Markdown(articles_df.loc[0, "markdown"]))

[김지현 기자 @이코노미톡뉴스] LG생활건강(이하 LG생건)이 2026년 3분기 뷰티 부문의 적자 폭을 줄이며 전년 동기 대비 50%가 넘는 영업이익 증가율을 기록했으나, 시장 기대치를 크게 밑도는 ‘어닝 쇼크’ 성적표를 받아들었다. 만성 적자 늪에 빠졌던 북미 자회사 에이본(The Avon Company)을 매각하며 재무 불확실성을 도려내는 결단을 내렸지만, 중국 시장의 구조적 침체와 캐시카우(현금창출원)였던 음료 부문의 수익성 악화가 겹치며 성장 동력이 둔화됐다는 평가다. 특히 같은 기간 아모레퍼시픽이 두 자릿수 영업이익률을 회복하고, 후발주자인 에이피알(APR)이 화장품 매출과 전사 영업이익에서 모두 LG생건을 추월하면서 국내 뷰티 업계의 지형도 변화가 한층 뚜렷해지고 있다.

**LG생건: 영업익 증가는 기저효과…"화장품 적자 지속·음료 수익성↓"**

---

금융감독원 전자공시시스템에 따르면 LG생건의 2026년 3분기 연결 기준 매출액은 1조 5,774억 원으로 전년 동기 대비 0.2% 감소했고, 영업이익은 704억 원으로 52.4% 증가했다. 표면적으로는 영업이익이 큰 폭으로 반등한 것처럼 보이지만, 이는 지난해 3분기 기저효과에 따른 결과로 증권가 컨센서스(전망치 평균 약 1,013억 원)를 30.5% 밑돈 수치다.

![LG생활건강은 2026년 3분기 영업이익이 전년 동기 대비 52.4% 증가한 704억 원을 기록하며 외형상 실적 개선에 성공했지만, 시장 기대치에는 크게 못 미치는 성적표를 받아들었다. 매출은 1조 5,774억 원으로 소폭 감소했고, 영업이익은 증권가 컨센서스를 약 30% 하회했다. [이미지=e톡뉴스 AI]](https://cdn.economytalk.kr/news/photo/202611/425107_233847_4527.png)

LG생활건강은 2026년 3분기 영업이익이 전년 동기 대비 52.4% 증가한 704억 원을 기록하며 외형상 실적 개선에 성공했지만, 시장 기대치에는 크게 못 미치는 성적표를 받아들었다. 매출은 1조 5,774억 원으로 소폭 감소했고, 영업이익은 증권가 컨센서스를 약 30% 하회했다. [이미지=e톡뉴스 AI]

직전 분기(2026년 2분기)와 비교하면 수익성 둔화는 더욱 두드러진다. 전분기 대비 매출은 4.8%, 영업이익은 31.5% 줄어들었으며 전사 영업이익률은 2분기 6.2%에서 3분기 4.5%로 하락했다. 여기에 북미 자회사 에이본 매각(9월부터 연결 제외)에 따른 일회성 처분손실 585억 원이 영업외비용으로 반영되면서 법인세비용차감전계속사업이익은 33억 원으로 전년 동기 대비 89.8% 급감했고, 당기순이익 역시 128억 원으로 45.2% 줄어들었다. 이 같은 실망감에 실적 발표 이튿날인 11월 4일 유가증권시장에서 LG생건 주가는 전 거래일 대비 11.83%(3만 1,000원) 급락한 23만 1,000원에 거래를 마쳤다

**아모레는 ‘턴어라운드’, 에이피알엔 ‘화장품 매출 역전’ 허용**

---

이번 3분기 실적에서 가장 뼈아픈 대목은 경쟁사들과의 수익성 및 본업 경쟁력 격차다. 글로벌 K-뷰티 호황 속에서 서구권 안착에 성공한 기업들과 구조조정 진통을 겪고 있는 LG생건의 성적표가 극명하게 엇갈렸다.

< K-뷰티 빅4의 Q3 실적 >

| **기업명** | **Q3 연결 매출(YoY)** | **Q3 영업익(YoY)** | **영업이익률** | **Q3 당기순익(YoY)** |
| --- | --- | --- | --- | --- |
| **LG생활건강** | 1조 5,774억 (-0.2%) | 704억 (+52.4%) | 4.5% | 128억 (-45.2%) |
| **아모레퍼시픽** | 1조 1,295 (+11.1%) | 1,152억 (+25.4%) | 10.2% | 585억 (-14.3%) |
| **에이피알(APR)** | 7,731억 (+100.3%) | 1,606억 (+68.9%) | 20.8% | 1,102억 (+47.6%) |
| **애경산업** | 1,904억 (+12.5%) | -33억 (적자전환) | -1.7% | -98억 (적자전환) |

![‘듀얼 리페어 리프트 M.D. 크림 미스트’. [사진=에이피 뷰티(AP BEAUTY) 제공]](https://cdn.economytalk.kr/news/photo/202611/425107_233846_4026.jpg)

‘듀얼 리페어 리프트 M.D. 크림 미스트’. [사진=에이피 뷰티(AP BEAUTY) 제공]

전체 외형에서는 음료와 생활용품을 거느린 LG생건이 1조 5,774억 원으로 1위를 지켰으나, 내실에서는 완전히 밀려났다. 아모레퍼시픽은 3분기 연결 매출 1조 1,295억 원(+11.1%), 영업이익 1,152억 원(+25.4%)을 기록하며 영업이익률 10.2%를 달성했다. 국내 사업 영업이익이 846억 원으로 42% 급증한 데다, 미주와 일본에서 에스트라·코스알엑스(COSRX)·설화수·헤라 등이 고성장하며 해외 영업이익도 485억 원(+14%)을 거뒀다.

더욱 충격적인 변화는 신흥 강자 에이피알(APR)의 약진이다. 에이피알은 해외 매출 비중이 94%까지 확대되며 3분기 매출 7,731억 원(+100.3%), 영업이익 1,606억 원(+68.9%, 영업이익률 20.8%)으로 창사 이래 최대 분기 매출을 새로 썼다. 특히 에이피알의 화장품·뷰티 단일 부문 매출은 6,823억 원(+150.6%)으로, LG생건의 전체 뷰티 부문 매출(6,608억 원)을 처음으로 앞질렀다. 전사 영업이익에서도 에이피알은 LG생건의 2.3배에 달하는 이익을 내며 K-뷰티 수익성 1위 자리를 차지했다.

![지난 9월 24~27일 멕시코 멕시코시티에서 열린 K-엑스포에 마련된 더페이스샵 홍보부스에 많은 인파가 몰렸다. [사진=LG H&H Co., Ltd 제공]](https://cdn.economytalk.kr/news/photo/202611/425107_233844_3258.jpg)

지난 9월 24~27일 멕시코 멕시코시티에서 열린 K-엑스포에 마련된 더페이스샵 홍보부스에 많은 인파가 몰렸다. [사진=LG H&H Co., Ltd 제공]

**LG생건: ‘에이본 손절·국내 효율화’ 빛났지만 ‘미·중 적자·음료 부진’**

---

LG생건의 3분기 실적을 사업부별로 살펴보면 구조조정 효과가 나타나기 시작했다는 점은 긍정적이다. 뷰티 부문 매출은 6,608억 원(-0.3% YoY)으로 전년 수준을 유지했으며, 영업손실은 지난해 486억 원에서 109억 원으로 크게 줄어 적자 규모를 377억 원 축소했다. 국내 백화점과 할인점 중심의 비효율 오프라인 채널을 정리하고 온라인·MBS(멀티브랜드숍)·면세 중심으로 영업 구조를 재편해 고정비를 낮춘 효과다. 특히 면세 매출이 145.5% 급증(925억 원)하며 주력 브랜드 더후가 성장세로 전환했고, 닥터그루트·도미나스·유시몰(오랄 뷰티)·VDL 등도 고성장을 이어갔으며, 만성 적자 사업이었던 북미 에이본 매각까지 마무리하면서 장기적인 손익 개선 기반을 마련했다.

전략 브랜드 육성 성과도 가시화되고 있다. 3분기 북미 전체 매출은 에이본 연결 제외 영향으로 1,355억 원(-12.5%)을 기록했으나, 에이본을 제외한 북미 자체 브랜드 매출은 약 5% 성장했다. 주력 헤어케어 브랜드 닥터그루트가 지난 8월 북미 세포라 오프라인 매장 400여 곳에 입점한 데 이어, 더페이스샵이 미국 월마트(약 1,600개 매장)와 얼타(ULTA, 약 500개 매장)에, 인디 색조 브랜드 힌스(hince)가 얼타(약 400개 매장)에 잇따라 입점하며 북미 리테일 접점을 넓혔다. 여기에 힌스를 운영하는 자회사 비바웨이브의 잔여 지분 25%를 콜옵션 행사로 추가 매입해 지분 100% 체제를 구축했고, 생활용품(HDB) 부문 역시 ‘피지 모락셀라(세탁세제)’, ‘46cm(체취케어)’, ‘홈스타(주거용품)’ 등 히어로 제품이 실적을 견인하며 매출 4,192억 원(+3.8%), 영업이익 310억 원(-3.1%)을 기록했다.

![최근 LG생활건강의 대표 K-뷰티 브랜드 ‘더페이스샵’이 미국 대형 화장품 유통업체 ‘얼타 뷰티(Ulta Beauty)’ 온·오프라인에 입점했다. (사진은) 더페이스샵 미감수 제품 라인업. [사진=LG Life & Health 제공]](https://cdn.economytalk.kr/news/photo/202611/425107_233845_3642.jpg)

최근 LG생활건강의 대표 K-뷰티 브랜드 ‘더페이스샵’이 미국 대형 화장품 유통업체 ‘얼타 뷰티(Ulta Beauty)’ 온·오프라인에 입점했다. (사진은) 더페이스샵 미감수 제품 라인업. [사진=LG Life & Health 제공]

반면 해외 핵심 시장의 적자 지속과 비화장품 사업 약세는 여전히 해결해야 할 과제다. 전체 해외 매출은 4,323억 원으로 12.5% 감소했으며, 특히 중국 매출이 온라인 사업 구조 재편 여파로 21.1% 급감한 1,151억 원에 그쳤다. 더후 중심의 중국 사업이 침체를 벗어나지 못하면서 고정비 부담에 따른 적자가 이어지고 있고, 북미 역시 신규 채널 입점 초기 단계의 마케팅 비용 투입이 늘어나며 외형 성장이 수익성으로 직결되지 못했다. 더욱이 화장품 부진을 방어하던 음료(리프레시먼트) 사업도 매출 4,974억 원(-3.1%), 영업이익 503억 원(-19.9%)으로 크게 둔화됐다. 내수 소비 침체와 원부자재 가격 상승, 제로 탄산 판촉 경쟁 심화가 겹치면서 LG생건은 중국 회복과 북미 안착, 음료 수익성 방어라는 세 가지 과제를 동시에 해결해야 하는 상황에 놓여 있다.

< 증권가 Q4 향후 전망 >

| **증권사** | **애널리스트** | **투자의견** | **목표주가 변동** |
| --- | --- | --- | --- |
| **메리츠증권** | 박신애·김건우 | 매수 유지 | 37만 → 32만 (-13.5%) |
| **한국투자증권** | 김명주 | 매수 유지 | 34만 → 31만 (-8.8%) |
| **키움증권** | 손예빈 | 매수→중립 | 33만 → 28만 (-15.2%) |
| **하나증권** | 박은정 | 매수→중립 | 32만 → 27만 (-15.6%) |
| **LS증권** | 오린아 | 매수→보유 | 35만 → 26만 (-25.7%) |
| **DB증권** | 허제나 | 매수 유지 | 36만 유지 |
| **한화투자증권** | 한유정 | 매수 유지 | 35만 유지 |
| **신한투자증권** | 박현진 | 매수 유지 | 33만 5,000원 유지 |

**“본격적인 턴어라운드는 2027년 이후?”**

---

3분기 어닝 쇼크 이후 주요 증권사들은 목표주가를 일제히 낮추며 당분간 보수적인 접근이 필요하다는 의견을 내놓고 있다. 한국투자증권과 메리츠증권은 생활용품을 제외한 뷰티·음료 사업의 회복 속도가 예상보다 더디고 전사 성장을 주도할 단기 모멘텀이 부족하다고 평가했다. 키움증권과 하나증권 역시 중국 사업 적자가 이어지는 가운데 북미 오프라인 유통망 확장에 따른 초기 비용 부담이 불가피해, 미국과 중국 시장에서 화장품 사업의 실질적인 수익성 개선이 확인되기 전까지는 시간이 필요하다며 투자의견을 ‘중립’으로 하향했다. 메리츠증권은 4분기 영업이익을 69억 원, 한화투자증권은 4분기 매출 1조 5,071억 원(+2.4%)과 영업이익 134억 원(흑자전환)으로 추정하며 계절적 비수기와 구조조정 비용 영향으로 4분기 이익 규모는 제한적일 것으로 내다봤다.

다만 체질 개선 방향성 자체는 긍정적이라는 평가도 공존한다. 한화투자증권, 신한투자증권, DB증권 등은 면세 채널의 수익 구조 개선과 에이본 매각을 통한 북미 사업 단순화, 그리고 닥터그루트·더페이스샵·힌스를 앞세운 북미·일본 채널 확장이 중장기 실적 반등의 발판이 될 것이라며 목표주가를 유지했다.

결국 시장의 관심은 2027년 이후 실질적인 턴어라운드 가능성에 집중되고 있다. LG생건이 K-뷰티 선두권 지위를 회복하기 위해서는 에이본 매각 이후 가벼워진 북미 법인에서 닥터그루트와 힌스가 초기 마케팅비를 상쇄하는 핵심 성장동력으로 자리 잡고, 중국 온라인 채널 재편을 조속히 마무리해 화장품 본업의 안정적인 흑자 구조를 구축해야 한다. 증권가가 공통적으로 제시하는 핵심 과제는 단순한 ‘적자 축소’를 넘어선 ‘지속 가능한 외형 성장과 수익성 회복’이며, 그 성과가 본격적으로 확인되는 시점은 2027년 상반기 이후가 될 전망이다.